# 2.4 Elementary Matrices

**Course:** Elementary Linear Algebra (Larson, 8e)  
**Chapter 2 — Matrices**

---

## 1. Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '../..'))

In [ ]:
from linalg_core.matrix import Matrix
from linalg_core.ops import (
    add, scalar_mul, multiply, transpose, inverse,
    elementary_swap, elementary_scale, elementary_add,
    lu_factorize, forward_sub, back_sub,
)
from linalg_core.elimination import swap_rows, to_rref, solve
from linalg_core import EPSILON


def matrices_equal(A, B):
    """Check if two matrices are equal within EPSILON tolerance."""
    if A.rows != B.rows or A.cols != B.cols:
        return False
    return all(abs(a - b) < EPSILON for a, b in zip(A.data, B.data))


def is_lower_triangular(M):
    """Check that M is lower triangular (zeros above diagonal)."""
    for i in range(M.rows):
        for j in range(i + 1, M.cols):
            if abs(M[i, j]) > EPSILON:
                return False
    return True


def is_upper_triangular(M):
    """Check that M is upper triangular (zeros below diagonal)."""
    for i in range(M.rows):
        for j in range(0, min(i, M.cols)):
            if abs(M[i, j]) > EPSILON:
                return False
    return True


def has_unit_diagonal(M):
    """Check that M has 1s on its main diagonal."""
    n = min(M.rows, M.cols)
    return all(abs(M[i, i] - 1.0) < EPSILON for i in range(n))


print("Setup complete.")

---

## 2. Motivation

Every row operation you performed in Chapter 1 — swapping rows, scaling a row,
adding a multiple of one row to another — is secretly a **matrix multiplication**.

The matrix that performs the row operation is called an **elementary matrix**.
It is obtained by applying that same row operation to the identity matrix $I$.

This insight leads directly to **LU decomposition**: instead of recording the
sequence of row operations, we factor a matrix as

$$A = LU$$

where $L$ is **lower triangular** (storing the multipliers) and $U$ is **upper
triangular** (the row-echelon form). To solve $A\mathbf{x} = \mathbf{b}$,
we make two easy triangular passes instead of one expensive RREF sweep.

When we need to solve the *same* system with many different right-hand sides
$\mathbf{b}_1, \mathbf{b}_2, \ldots$, the LU factorization pays for itself:
we factor once and substitute many times.

---

## 3. Build — Core Concepts

### 3.1 Elementary Matrices

An **elementary matrix** is an identity matrix with exactly one row operation
applied. There are three types, matching the three row operations:

| Row operation | Elementary matrix | Notation |
|---|---|---|
| Swap rows $i$ and $j$ | $E_{\text{swap}}(i, j)$ | `elementary_swap(n, i, j)` |
| Scale row $i$ by $c \neq 0$ | $E_{\text{scale}}(i, c)$ | `elementary_scale(n, i, c)` |
| Add $c \times$ row $j$ to row $i$ | $E_{\text{add}}(i, j, c)$ | `elementary_add(n, i, j, c)` |

> **Definition (Larson, Sec. 2.4).** An $n \times n$ matrix is **elementary** if
> it can be obtained from $I_n$ by a single elementary row operation.

Let's build each type and inspect its structure.

In [ ]:
n = 3
I3 = Matrix.identity(n)
print("Identity I_3:")
print(I3)

print("\n--- Type 1: Row Swap ---")
E_swap = elementary_swap(n, 0, 2)
print("E_swap(3, 0, 2) — swap rows 0 and 2:")
print(E_swap)

print("\n--- Type 2: Row Scaling ---")
E_scale = elementary_scale(n, 1, 5)
print("E_scale(3, 1, 5) — scale row 1 by 5:")
print(E_scale)

print("\n--- Type 3: Row Addition ---")
E_add = elementary_add(n, 2, 0, -3)
print("E_add(3, 2, 0, -3) — add -3 * row 0 to row 2:")
print(E_add)

### 3.2 Row Operation = Matrix Multiplication

> **Theorem (Larson, Sec. 2.4).** If $E$ is the elementary matrix obtained by
> performing a row operation on $I_n$, then performing that same row operation
> on an $n \times n$ matrix $A$ is equivalent to computing $EA$.

In other words, left-multiplying $A$ by $E$ **is** the row operation.

Let's verify this for all three types. We take a matrix $A$, perform a row
operation directly, and confirm that $EA$ gives the same result.

In [ ]:
A = Matrix.from_lists([
    [1, 2, 3],
    [4, 5, 6],
    [7, 8, 9],
])
print("A ="); print(A)

print("\n=== Type 1: Swap rows 0 and 2 ===")
E1 = elementary_swap(3, 0, 2)
EA1 = multiply(E1, A)
A_copy1 = A.copy()
swap_rows(A_copy1, 0, 2)
print("E * A ="); print(EA1)
print("Direct swap ="); print(A_copy1)
print(f"Match: {matrices_equal(EA1, A_copy1)}")

print("\n=== Type 2: Scale row 1 by 5 ===")
E2 = elementary_scale(3, 1, 5)
EA2 = multiply(E2, A)
A_copy2 = A.copy()
for k in range(A_copy2.cols):
    A_copy2[1, k] = A_copy2[1, k] * 5
print("E * A ="); print(EA2)
print("Direct scale ="); print(A_copy2)
print(f"Match: {matrices_equal(EA2, A_copy2)}")

print("\n=== Type 3: Add -3 * row 0 to row 2 ===")
E3 = elementary_add(3, 2, 0, -3)
EA3 = multiply(E3, A)
A_copy3 = A.copy()
for k in range(A_copy3.cols):
    A_copy3[2, k] = A_copy3[2, k] + (-3) * A_copy3[0, k]
print("E * A ="); print(EA3)
print("Direct add ="); print(A_copy3)
print(f"Match: {matrices_equal(EA3, A_copy3)}")

### 3.3 Invertibility and Elementary Matrices

Every elementary matrix is **invertible**, because every row operation can be
undone:

| Elementary matrix | Its inverse |
|---|---|
| $E_{\text{swap}}(i,j)$ | $E_{\text{swap}}(i,j)$ — swap again |
| $E_{\text{scale}}(i,c)$ | $E_{\text{scale}}(i, 1/c)$ — scale by $1/c$ |
| $E_{\text{add}}(i,j,c)$ | $E_{\text{add}}(i,j,-c)$ — subtract instead of add |

> **Theorem (Larson, Sec. 2.4).** A square matrix $A$ is invertible if and only
> if it is **row-equivalent** to $I$. Equivalently, there exist elementary
> matrices $E_1, E_2, \ldots, E_k$ such that
>
> $$A = E_1 E_2 \cdots E_k$$

This says that every invertible matrix is a product of elementary matrices.
Each $E_k$ encodes one step of the row reduction that transforms $I$ into $A$.

In [ ]:
print("=== Inverses of elementary matrices ===")

E_sw = elementary_swap(3, 0, 2)
E_sw_inv = elementary_swap(3, 0, 2)
prod_sw = multiply(E_sw, E_sw_inv)
print("Swap * Swap ="); print(prod_sw)
print(f"= I? {matrices_equal(prod_sw, Matrix.identity(3))}")

print()
E_sc = elementary_scale(3, 1, 5)
E_sc_inv = elementary_scale(3, 1, 1/5)
prod_sc = multiply(E_sc, E_sc_inv)
print("Scale(5) * Scale(1/5) ="); print(prod_sc)
print(f"= I? {matrices_equal(prod_sc, Matrix.identity(3))}")

print()
E_ad = elementary_add(3, 2, 0, -3)
E_ad_inv = elementary_add(3, 2, 0, 3)
prod_ad = multiply(E_ad, E_ad_inv)
print("Add(-3) * Add(+3) ="); print(prod_ad)
print(f"= I? {matrices_equal(prod_ad, Matrix.identity(3))}")

print("\n=== Expressing A as a product of elementary matrices ===")
print("If row operations E3 E2 E1 A = I, then A = E1^{-1} E2^{-1} E3^{-1}.")
print("\nExample: reduce A to I, then reverse the operations.")

A = Matrix.from_lists([
    [1, 0, 1],
    [0, 1, 0],
    [2, 0, 1],
])
print("\nA ="); print(A)

E1 = elementary_add(3, 2, 0, -2)
step1 = multiply(E1, A)
print("\nE1 * A (R2 -= 2*R0):")
print(step1)

E2 = elementary_scale(3, 2, -1)
step2 = multiply(E2, step1)
print("\nE2 * E1 * A (scale R2 by -1):")
print(step2)

E3 = elementary_add(3, 0, 2, -1)
step3 = multiply(E3, step2)
print("\nE3 * E2 * E1 * A (R0 -= R2):")
print(step3)
print(f"= I? {matrices_equal(step3, Matrix.identity(3))}")

E1_inv = elementary_add(3, 2, 0, 2)
E2_inv = elementary_scale(3, 2, -1)
E3_inv = elementary_add(3, 0, 2, 1)
product = multiply(E1_inv, multiply(E2_inv, E3_inv))
print("\nE1^{-1} * E2^{-1} * E3^{-1} ="); print(product)
print(f"= A? {matrices_equal(product, A)}")

### 3.4 LU Decomposition — The Concept

When we perform Gaussian elimination on $A$ (without row swaps), the
elementary matrices used are all of the **row-addition** type $E_{\text{add}}$.
Their product $E_k \cdots E_2 E_1$ transforms $A$ into an upper triangular
matrix $U$:

$$E_k \cdots E_2 E_1 \, A = U$$

Solving for $A$:

$$A = E_1^{-1} E_2^{-1} \cdots E_k^{-1} \, U = LU$$

The matrix $L = E_1^{-1} E_2^{-1} \cdots E_k^{-1}$ is **lower triangular**
with **1s on the diagonal**. Its below-diagonal entries are exactly the
**multipliers** used during elimination.

$$A = \underbrace{L}_{\text{lower triangular}} \cdot \underbrace{U}_{\text{upper triangular}}$$

**Why this matters:** To solve $A\mathbf{x} = \mathbf{b}$, we rewrite it as
$LU\mathbf{x} = \mathbf{b}$ and solve in two steps:

1. **Forward substitution:** Solve $L\mathbf{y} = \mathbf{b}$ for $\mathbf{y}$
2. **Back substitution:** Solve $U\mathbf{x} = \mathbf{y}$ for $\mathbf{x}$

Both steps are $O(n^2)$, compared to $O(n^3)$ for full RREF. Once we have $L$
and $U$, solving with a new $\mathbf{b}$ is cheap.

### 3.5 Doolittle's Method — `lu_factorize(A)`

**Doolittle's algorithm** computes $L$ and $U$ simultaneously by marching
down each column and recording the multiplier $\ell_{ij} = a_{ij}^{(k)}/a_{jj}^{(k)}$
in $L$ while performing the elimination in $U$.

Let's factor

$$A = \begin{bmatrix} 2 & -1 & 1 \\ 4 & 1 & -1 \\ -2 & 3 & 1 \end{bmatrix}$$

In [ ]:
A = Matrix.from_lists([
    [2, -1,  1],
    [4,  1, -1],
    [-2, 3,  1],
])
print("A ="); print(A)

L, U = lu_factorize(A)

print("\nL ="); print(L)
print("\nU ="); print(U)

print("\n--- Structure checks ---")
print(f"L is lower triangular: {is_lower_triangular(L)}")
print(f"L has 1s on diagonal:  {has_unit_diagonal(L)}")
print(f"U is upper triangular: {is_upper_triangular(U)}")

LU = multiply(L, U)
print("\nL * U ="); print(LU)
print(f"\nL * U == A? {matrices_equal(LU, A)}")

**Reading $L$:** The sub-diagonal entries of $L$ are the multipliers from
elimination.

- Column 0: To eliminate $A[1,0] = 4$, we used multiplier $4/2 = 2$, so $L[1,0] = 2$.
  To eliminate $A[2,0] = -2$, we used multiplier $-2/2 = -1$, so $L[2,0] = -1$.
- Column 1: After the first elimination pass, the $(2,1)$ entry becomes $2$.
  The pivot is $3$. So $L[2,1] = 2/3$.

The **1s on the diagonal** of $L$ come from the fact that each row operation
$E_{\text{add}}$ has 1s on its diagonal, and so does each inverse.

### 3.6 Forward Substitution — Solve $L\mathbf{y} = \mathbf{b}$

Given a lower triangular system

$$\begin{bmatrix}
\ell_{00} & 0 & 0 \\
\ell_{10} & \ell_{11} & 0 \\
\ell_{20} & \ell_{21} & \ell_{22}
\end{bmatrix}
\begin{bmatrix} y_0 \\ y_1 \\ y_2 \end{bmatrix}
=
\begin{bmatrix} b_0 \\ b_1 \\ b_2 \end{bmatrix}$$

we solve top-to-bottom:

$$y_i = \frac{1}{\ell_{ii}}\left(b_i - \sum_{j=0}^{i-1} \ell_{ij}\, y_j\right)$$

Since $L$ has 1s on the diagonal, the division is trivial.

In [ ]:
b = [4, 2, 6]
print(f"Solving Ly = b where b = {b}")
print("\nL ="); print(L)

y = forward_sub(L, b)
print(f"\ny = {y}")

print("\n--- Verify: L * y should equal b ---")
y_vec = Matrix.from_vector(y)
Ly = multiply(L, y_vec)
Ly_list = [Ly[i, 0] for i in range(Ly.rows)]
print(f"L * y = {Ly_list}")
print(f"b     = {b}")
match = all(abs(a - b_) < EPSILON for a, b_ in zip(Ly_list, b))
print(f"Match: {match}")

### 3.7 Back Substitution — Solve $U\mathbf{x} = \mathbf{y}$

Given an upper triangular system

$$\begin{bmatrix}
u_{00} & u_{01} & u_{02} \\
0 & u_{11} & u_{12} \\
0 & 0 & u_{22}
\end{bmatrix}
\begin{bmatrix} x_0 \\ x_1 \\ x_2 \end{bmatrix}
=
\begin{bmatrix} y_0 \\ y_1 \\ y_2 \end{bmatrix}$$

we solve bottom-to-top:

$$x_i = \frac{1}{u_{ii}}\left(y_i - \sum_{j=i+1}^{n-1} u_{ij}\, x_j\right)$$

In [ ]:
print(f"Solving Ux = y where y = {y}")
print("\nU ="); print(U)

x = back_sub(U, y)
print(f"\nx = {x}")

print("\n--- Verify: U * x should equal y ---")
x_vec = Matrix.from_vector(x)
Ux = multiply(U, x_vec)
Ux_list = [Ux[i, 0] for i in range(Ux.rows)]
print(f"U * x = {Ux_list}")
print(f"y     = {y}")
match = all(abs(a - b_) < EPSILON for a, b_ in zip(Ux_list, y))
print(f"Match: {match}")

### 3.8 LU Solve — Two Passes Instead of RREF

Putting it all together, to solve $A\mathbf{x} = \mathbf{b}$:

1. Factor $A = LU$ (one-time cost, $O(n^3)$)
2. Forward-substitute: $L\mathbf{y} = \mathbf{b}$ ($O(n^2)$)
3. Back-substitute: $U\mathbf{x} = \mathbf{y}$ ($O(n^2)$)

Let's solve $A\mathbf{x} = \mathbf{b}$ where $\mathbf{b} = [4, 2, 6]^T$
and verify the result matches the RREF approach.

In [ ]:
A = Matrix.from_lists([
    [2, -1,  1],
    [4,  1, -1],
    [-2, 3,  1],
])
b = [4, 2, 6]

print("="*50)
print("METHOD 1: LU Decomposition")
print("="*50)

L, U = lu_factorize(A)
y = forward_sub(L, b)
x_lu = back_sub(U, y)

print(f"L ="); print(L)
print(f"\nU ="); print(U)
print(f"\nForward sub: y = {y}")
print(f"Back sub:    x = {x_lu}")

print("\n" + "="*50)
print("METHOD 2: RREF")
print("="*50)

aug = Matrix(3, 4)
for i in range(3):
    for j in range(3):
        aug[i, j] = A[i, j]
    aug[i, 3] = b[i]

sol_type, x_rref = solve(aug)
print(f"Solution type: {sol_type}")
print(f"x = {x_rref}")

print("\n" + "="*50)
print("COMPARISON")
print("="*50)
match = all(abs(a - b_) < EPSILON for a, b_ in zip(x_lu, x_rref))
print(f"LU solution:   {x_lu}")
print(f"RREF solution: {x_rref}")
print(f"Match: {match}")

print("\n--- Verify: A * x should equal b ---")
x_vec = Matrix.from_vector(x_lu)
Ax = multiply(A, x_vec)
Ax_list = [Ax[i, 0] for i in range(Ax.rows)]
print(f"A * x = {Ax_list}")
print(f"b     = {b}")
verify = all(abs(a - b_) < EPSILON for a, b_ in zip(Ax_list, b))
print(f"A * x == b? {verify}")

In [ ]:
print("="*50)
print("LU Advantage: Multiple right-hand sides")
print("="*50)
print("\nReusing the SAME L, U factorization for three different b vectors:\n")

b_vectors = [
    [4, 2, 6],
    [1, 0, 0],
    [0, 1, -1],
]

for idx, b_i in enumerate(b_vectors):
    y_i = forward_sub(L, b_i)
    x_i = back_sub(U, y_i)
    x_vec_i = Matrix.from_vector(x_i)
    Ax_i = multiply(A, x_vec_i)
    Ax_list_i = [Ax_i[r, 0] for r in range(Ax_i.rows)]
    ok = all(abs(a - b_) < EPSILON for a, b_ in zip(Ax_list_i, b_i))
    print(f"  b_{idx+1} = {b_i}  =>  x = [{', '.join(f'{v:.4f}' for v in x_i)}]  Ax==b? {ok}")

---

## 4. Verify — Systematic Checks

We verify the key invariants:

1. $L \cdot U = A$
2. $L$ is lower triangular with 1s on the diagonal
3. $U$ is upper triangular
4. The LU solve matches the RREF solve

In [ ]:
import random
random.seed(42)

print("="*60)
print("VERIFICATION: LU Factorization on random 4x4 matrices")
print("="*60)

num_pass = 0
num_total = 10

for trial in range(num_total):
    data = [random.uniform(-10, 10) for _ in range(16)]
    M = Matrix(4, 4, data)

    try:
        L_t, U_t = lu_factorize(M)
    except ValueError:
        print(f"  Trial {trial+1}: skipped (zero pivot — needs pivoting)")
        num_total -= 1
        continue

    check_product = matrices_equal(multiply(L_t, U_t), M)
    check_lower = is_lower_triangular(L_t)
    check_diag = has_unit_diagonal(L_t)
    check_upper = is_upper_triangular(U_t)

    ok = check_product and check_lower and check_diag and check_upper
    if ok:
        num_pass += 1
    status = "PASS" if ok else "FAIL"
    print(f"  Trial {trial+1}: [{status}] LU=A:{check_product} L_lower:{check_lower} L_diag1:{check_diag} U_upper:{check_upper}")

print(f"\nPassed: {num_pass}/{num_total}")

In [ ]:
print("="*60)
print("VERIFICATION: LU solve vs RREF solve")
print("="*60)

random.seed(99)
all_match = True

for trial in range(10):
    data = [random.uniform(-10, 10) for _ in range(9)]
    M = Matrix(3, 3, data)
    b_rand = [random.uniform(-10, 10) for _ in range(3)]

    try:
        L_t, U_t = lu_factorize(M)
    except ValueError:
        print(f"  Trial {trial+1}: skipped (zero pivot)")
        continue

    y_t = forward_sub(L_t, b_rand)
    x_lu_t = back_sub(U_t, y_t)

    aug_t = Matrix(3, 4)
    for i in range(3):
        for j in range(3):
            aug_t[i, j] = M[i, j]
        aug_t[i, 3] = b_rand[i]
    sol_type_t, x_rref_t = solve(aug_t)

    if sol_type_t == "unique":
        match = all(abs(a - b_) < EPSILON for a, b_ in zip(x_lu_t, x_rref_t))
    else:
        match = False

    status = "PASS" if match else "FAIL"
    if not match:
        all_match = False
    print(f"  Trial {trial+1}: [{status}] LU={[round(v,4) for v in x_lu_t]} RREF={[round(v,4) for v in x_rref_t] if x_rref_t else 'N/A'}")

print(f"\nAll match: {all_match}")

In [ ]:
print("="*60)
print("VERIFICATION: Elementary matrix properties")
print("="*60)

I4 = Matrix.identity(4)

print("\n--- Swap matrices are self-inverse ---")
for (i, j) in [(0,1), (0,3), (1,2), (2,3)]:
    E = elementary_swap(4, i, j)
    prod = multiply(E, E)
    ok = matrices_equal(prod, I4)
    print(f"  swap({i},{j}): E*E = I? {ok}")

print("\n--- Scale(c) * Scale(1/c) = I ---")
for (i, c) in [(0, 3), (1, -2), (2, 0.5), (3, 7)]:
    E = elementary_scale(4, i, c)
    E_inv = elementary_scale(4, i, 1/c)
    prod = multiply(E, E_inv)
    ok = matrices_equal(prod, I4)
    print(f"  scale(row={i}, c={c}): E*E_inv = I? {ok}")

print("\n--- Add(c) * Add(-c) = I ---")
for (t, s, c) in [(1, 0, 3), (2, 0, -5), (3, 1, 2), (0, 3, -4)]:
    E = elementary_add(4, t, s, c)
    E_inv = elementary_add(4, t, s, -c)
    prod = multiply(E, E_inv)
    ok = matrices_equal(prod, I4)
    print(f"  add(target={t}, source={s}, c={c}): E*E_inv = I? {ok}")

---

## 5. Exercises

Test your understanding with the following problems. Write code in the provided cells.

### Exercise 1 (Easy)

Write the $4 \times 4$ elementary matrix that performs the row operation
"add $-5$ times row 1 to row 3." Then verify it by:

1. Building the matrix with `elementary_add`.
2. Applying it to

$$A = \begin{bmatrix} 1 & 0 & 0 & 0 \\ 0 & 2 & 0 & 0 \\ 0 & 0 & 3 & 0 \\ 0 & 0 & 0 & 4 \end{bmatrix}$$

3. Confirming that only row 3 changed, and it now equals the original
   row 3 plus $(-5) \times$ row 1.

In [ ]:
# YOUR CODE HERE


### Exercise 2 (Medium)

Compute the LU factorization of

$$B = \begin{bmatrix} 1 & 2 & 4 \\ 3 & 8 & 14 \\ 2 & 6 & 13 \end{bmatrix}$$

Then:

1. Verify that $L \cdot U = B$.
2. Verify the structural properties of $L$ and $U$.
3. Use the LU factorization to solve $B\mathbf{x} = \begin{bmatrix} 2 \\ 4 \\ 6 \end{bmatrix}$.
4. Confirm your answer by checking $B\mathbf{x} = \mathbf{b}$.

In [ ]:
# YOUR CODE HERE


### Exercise 3 (Challenge)

**Comparing computational cost: RREF vs LU.**

Suppose you must solve $A\mathbf{x} = \mathbf{b}$ for **three** different
right-hand sides $\mathbf{b}_1, \mathbf{b}_2, \mathbf{b}_3$ with the same
$n \times n$ matrix $A$.

**Method A — RREF each time:** Form $[A | \mathbf{b}_k]$ and row-reduce.
Each solve costs roughly $\frac{2}{3}n^3$ flops. Total: $3 \times \frac{2}{3}n^3 = 2n^3$.

**Method B — LU once, substitute three times:** Factor $A = LU$ for
$\frac{2}{3}n^3$ flops. Each forward/back substitution costs $n^2$ flops.
Total: $\frac{2}{3}n^3 + 3 \times 2n^2 = \frac{2}{3}n^3 + 6n^2$.

1. For $n = 100$, compute the estimated flop counts for both methods.
   What is the ratio?
2. Time both approaches using Python's `time` module on a random
   $100 \times 100$ system with 3 right-hand sides. Does the measured
   speedup match your theoretical prediction?

*Hint:* Use `time.perf_counter()` for accurate timing.

In [ ]:
# YOUR CODE HERE


---

## Summary

| Concept | Key Takeaway |
|---|---|
| **Elementary matrix** | An identity matrix with one row operation applied |
| **Row op = multiply** | Performing a row operation on $A$ is the same as left-multiplying by the corresponding elementary matrix $E$ |
| **Invertibility** | Every elementary matrix is invertible; $A$ is invertible iff $A = E_1 E_2 \cdots E_k$ |
| **LU decomposition** | $A = LU$ where $L$ is lower triangular (multipliers) and $U$ is upper triangular (REF) |
| **Forward substitution** | Solve $L\mathbf{y} = \mathbf{b}$ top-to-bottom in $O(n^2)$ |
| **Back substitution** | Solve $U\mathbf{x} = \mathbf{y}$ bottom-to-top in $O(n^2)$ |
| **LU solve** | Factor once ($O(n^3)$), substitute per RHS ($O(n^2)$) — huge savings for multiple $\mathbf{b}$ |

**Key equation:**

$$A\mathbf{x} = \mathbf{b} \;\Longrightarrow\; LU\mathbf{x} = \mathbf{b} \;\Longrightarrow\; \underbrace{L\mathbf{y} = \mathbf{b}}_{\text{forward sub}} \;\text{then}\; \underbrace{U\mathbf{x} = \mathbf{y}}_{\text{back sub}}$$